# Phase 6 — which network, measured rather than assumed

This notebook has run twice, with different runs in it. Each is reproducible
from the commit it cloned.

**First round (Kaggle Versions 1 to 3, commit `2418588`, sweep fix `191ee44`).**
Four networks at 1024 that differ in nothing else: *a* U-Net on ResNet-34, *b*
U-Net on ConvNeXt-Tiny, *c* UPerNet on ConvNeXt-Tiny, *d* Segformer on MiT-B2.
None beat the incumbent, and the reason was measured afterwards: ConvNeXt and
MiT expose no feature map at half resolution, and at 1024 that missing level is
where a filament a few pixels wide lives.

**Second round, first half (Kaggle Version 4, commit `1679966`).** Three runs,
each changing one thing against *a*:

| run | changes | PQ | against *a* | what moved |
|---|---|---|---|---|
| g | union of every annotator as the target | 0.3633 | -0.0095 | 40 more matches, 145 more false positives |
| f | 2048 at batch 1 | 0.3782 | +0.0054 | SQ 0.6677, the first run past 0.6555 |
| e | tu-hrnet_w32 at batch 2 | 0.3815 | +0.0087 | RQ, 58 fewer false positives |

**This version: run h, e and f stacked.** HRNet at 1536, batch 1. e and f won
on different parts of PQ, detection and mask shape, so if the two causes are
independent both should survive together. 1536 rather than 2048 because a
batch of one HRNet frame at 2048 holds twice the pixels e trained on, and e was
already held to a batch of two at 1024 to fit a T4.

Only h is trained. **Attach Version 4's output as an input** and e, f and g
are restored, scored from their saved results and kept as candidates; without
it they are skipped, never retrained. Their scoring in section 3 and their
sweeps are read back rather than recomputed.

The run is scored like the others: threshold 0.5, minimum area 400, rejoining
within 24 pixels at 1024, scaled into each run's map.

**Section 7 writes `submission.csv`** from whichever candidate scores best at
that shared setting, and checks it for overlapping masks before anything else
can be done with it. The model is the fold 0 one, trained on four fifths of the
training frames.

**Before running**, in the notebook settings:

1. Accelerator: **GPU T4 x2**
2. Internet: **on**
3. Add the competition data as an input
4. **Save Version** with *Save output* on, or the checkpoints and maps are discarded

One Run All trains, scores, writes probability maps and sweeps the
post-processing. Expected wall clock: three and a half to four and a half hours, of which the
sweep of h is about forty minutes. The sweep needs no GPU, so that
hour and a half is quota spent on an idle card; setting `SWEEP_WITH_GPU` to
`False` in section 6 leaves it for a later run with the accelerator off, which
restores everything else from this run's output and only sweeps.

**Re-running after one of them failed.** Attach the earlier version's output as
an input. Whatever it finished is copied back and skipped, so recovering one
crashed run costs only that run.

## 1. Clone the repository and put it on the path

The repository is cloned rather than installed, because `configs/paths.yaml`
and the frozen splits in `configs/splits/` sit beside the package rather than
inside it. Set `REF` to the branch or commit this run is to be reproducible
from.

In [ ]:
import sys

REPO_URL = "https://github.com/KeiichiIto1978/Solar_Filament_Segmentation_Challenge_2026.git"
REF = "feat/phase6-round2"  # branch or commit hash
CHECKOUT = "/kaggle/working/repo"

!rm -rf {CHECKOUT}
!git clone -q --branch {REF} {REPO_URL} {CHECKOUT}
!cd {CHECKOUT} && git log --oneline -1

# Only what Kaggle does not already have; torch stays as it is.
!pip install -q "segmentation-models-pytorch>=0.5"

if f"{CHECKOUT}/src" not in sys.path:
    sys.path.insert(0, f"{CHECKOUT}/src")

In [ ]:
import segmentation_models_pytorch as smp
import torch

import filament

print("filament", filament.__version__)
print("smp", smp.__version__)
print("torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.device_count())
if torch.cuda.is_available():
    print("name:", torch.cuda.get_device_name(0))

## 2. Point the package at the competition data

In [ ]:
import os
from pathlib import Path

from filament.paths import load_paths

ANNOTATION_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

candidates = sorted(Path("/kaggle/input").glob(f"**/{ANNOTATION_NAME}"))
if not candidates:
    raise SystemExit(
        "Could not find the annotation file under /kaggle/input. "
        "Attach the competition data to this notebook first."
    )

dataset_root = candidates[0].parent.parent
os.environ["MAGFILO_ROOT"] = str(dataset_root)
print("MAGFILO_ROOT =", dataset_root)

paths = load_paths().require_dataset()
print("train images:", len(list(paths.train_images.glob("*.jpeg"))))

## 3. The three runs

Scoring uses the post-processing Phase 3 settled on — threshold 0.5, minimum
area 400, rejoining within 24 pixels at 1024 — so that these numbers sit beside
the first round's. The ground truth is encoded once and handed to every
scoring call.

In [ ]:
from dataclasses import replace

from filament.data.coco import load_annotations
from filament.data.split import load_fold
from filament.postprocess.join import DEFAULT_MAX_OFFSET
from filament.training.config import TrainConfig

# Trained here.
TRAINED = ["h_unet_hrnet_1536"]
# Restored from Version 4's output when it is attached, never trained here.
EARLIER = ["e_unet_hrnet", "f_unet_resnet34_2048", "g_unet_resnet34_union"]
RUNS = EARLIER + TRAINED

# The configuration Phase 3 settled on, applied identically to every run.
SCORING = {"threshold": 0.5, "min_area": 400, "join_gap": 24.0}
# The resolution the rejoining distances above were chosen at.
SCORING_SIZE = 1024


def scoring_for(config):
    """SCORING, expressed in pixels of this run's probability map.

    The rejoining distances are counted in pixels of the map, so a run whose
    map is twice as fine needs them doubled to mean the same distance on the
    Sun. The minimum area is in pixels of the full frame and never changes.
    At 1024 this is exactly SCORING plus the default axis offset.
    """
    scale = config.image_size / SCORING_SIZE
    return SCORING | {
        "join_gap": SCORING["join_gap"] * scale,
        "join_offset": DEFAULT_MAX_OFFSET * scale,
    }


dataset = load_annotations(paths.train_annotations)
val_stems = load_fold(0, f"{CHECKOUT}/configs/splits").val
print(f"fold 0: {len(val_stems)} validation frames")

configs = {}
for name in RUNS:
    configs[name] = replace(
        TrainConfig.from_yaml(f"{CHECKOUT}/configs/phase6/{name}.yaml"),
        num_workers=2,
        output_dir=Path(f"/kaggle/working/{name}"),
    )
    config = configs[name]
    print(
        f"{name:<24} {config.architecture:<10} {config.encoder:<16} "
        f"{config.image_size}px batch {config.batch_size} "
        f"union {config.union_targets}  scoring {scoring_for(config)}"
    )

In [ ]:
import shutil

from filament.submit.rle import masks_to_gt_df, read_submission, write_submission

gt_path = Path("/kaggle/working/gt_fold0.csv")
if not gt_path.exists():
    # An earlier version of this notebook, attached as an input, has it already.
    found = sorted(Path("/kaggle/input").glob("**/gt_fold0.csv"))
    if found:
        shutil.copy(found[0], gt_path)
        print(f"ground truth restored from {found[0]}")

if gt_path.exists():
    gt_df = read_submission(gt_path)
else:
    gt_df = masks_to_gt_df(dataset, val_stems)
    write_submission(gt_df, gt_path)
print(f"{len(gt_df)} ground-truth filaments")

In [ ]:
import json
import logging
import time

import numpy as np

from filament.data.image import load_grayscale
from filament.evaluation import evaluate, predict_probability
from filament.training.loop import HISTORY_NAME, load_checkpoint, train

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# Everything here runs on whatever is attached. With the earlier version's work
# among the inputs there is nothing left to train, and the accelerator can be
# turned off; without it, this needs a GPU to finish in hours rather than days.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

EVAL_NAME = "eval_fold0.json"


def training_record(output_dir):
    """Minutes and best validation loss, from the history a run left behind."""
    history_path = output_dir / HISTORY_NAME
    if not history_path.exists():
        return None, None
    history = json.loads(history_path.read_text())
    minutes = round(sum(item["seconds"] for item in history) / 60, 1)
    return minutes, round(min(item["val_loss"] for item in history), 4)


def write_maps(model, name):
    maps_dir = Path(f"/kaggle/working/prob_fold0_{name}")
    maps_dir.mkdir(parents=True, exist_ok=True)
    if len(list(maps_dir.glob("*.npy"))) < len(val_stems):
        for stem in val_stems:
            probability = predict_probability(
                model,
                load_grayscale(paths.train_images / f"{stem}.jpeg"),
                size=configs[name].image_size,
                device=DEVICE,
            )
            np.save(maps_dir / f"{stem}.npy", probability.astype(np.float16))
    written = sorted(maps_dir.glob("*.npy"))
    print(f"{name}: {len(written)} maps, {sum(i.stat().st_size for i in written) / 1e6:.0f} MB")


def train_and_score(name):
    # Train one configuration if it is not already trained, then score it and
    # write its probability maps.
    config = configs[name]
    checkpoint = config.output_dir / "best.pt"
    maps_dir = Path(f"/kaggle/working/prob_fold0_{name}")

    # Scored by an earlier version of this notebook with its maps in place:
    # scoring again would spend GPU minutes to print the same numbers.
    eval_path = config.output_dir / EVAL_NAME
    if eval_path.exists() and len(list(maps_dir.glob("*.npy"))) >= len(val_stems):
        summary = json.loads(eval_path.read_text())
        print(f"{name}: already scored, PQ {summary['pq']}")
        return summary

    # Already trained, here or by the parallel step above: a session that died
    # part-way should not cost the runs that finished. Delete the directory to
    # force a retrain.
    if checkpoint.exists():
        print(f"{name}: reusing the checkpoint already in {checkpoint.parent}")
        training_minutes, best_val_loss = training_record(config.output_dir)
        best_epoch = torch.load(checkpoint, map_location="cpu", weights_only=False)["epoch"]
    elif DEVICE == "cpu":
        # The CPU pass only sweeps. Training here would take days, so a missing
        # checkpoint means the GPU pass's output is not attached.
        raise FileNotFoundError(
            f"No checkpoint for {name}. Attach the GPU pass's output as an input."
        )
    else:
        started = time.perf_counter()
        outcome = train(config)
        training_minutes = round((time.perf_counter() - started) / 60, 1)
        best_epoch = outcome.best_epoch
        best_val_loss = round(outcome.best_val_loss, 4)
        checkpoint = outcome.checkpoint

    model, _ = load_checkpoint(checkpoint)
    scoring = scoring_for(config)
    evaluation, _ = evaluate(
        model,
        dataset,
        paths.train_images,
        val_stems,
        size=config.image_size,
        device=DEVICE,
        gt_df=gt_df,
        **scoring,
    )
    summary = evaluation.to_dict() | {
        "architecture": config.architecture,
        "encoder": config.encoder,
        "image_size": config.image_size,
        "batch_size": config.batch_size,
        "union_targets": config.union_targets,
        "scoring": scoring,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "training_minutes": training_minutes,
    }
    eval_path.write_text(json.dumps(summary, indent=2))
    print(evaluation, f"| {training_minutes} min")

    # Written now rather than after every run: if the session ends early, the
    # runs that finished still leave behind what a sweep needs.
    write_maps(model, name)

    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return summary


# Kaggle empties /kaggle/working when a session starts, so an earlier version's
# work is only reachable through the inputs. Copying it back is what lets Run All
# finish a partly-done comparison rather than repeating the hours that succeeded.
for folder in list(RUNS) + [f"prob_fold0_{name}" for name in RUNS]:
    destination = Path("/kaggle/working") / folder
    if destination.exists() and any(destination.iterdir()):
        continue
    found = [item for item in Path("/kaggle/input").glob(f"**/{folder}") if item.is_dir()]
    if found:
        shutil.copytree(found[0], destination, dirs_exist_ok=True)
        print(f"{folder}: restored from {found[0]}")

### Training

Each untrained run is handed to `scripts/train.py` in its own process, pinned to
one card with `CUDA_VISIBLE_DEVICES`. Processes rather than threads: each seeds
its own random state, so a run's initial weights do not depend on anything else
this session did. The earlier rounds used both cards at once; this one has a
single run to train.

Without a card nothing is trained. A run that fails here is not retried here;
the scoring loop below finds no checkpoint for it and trains it in-process, so
the notebook still reaches its end.

In [ ]:
import os
import subprocess
import threading

gpu_count = torch.cuda.device_count()
# Without a card this is the CPU pass: nothing is trained, only swept.
QUEUES = {}
# One run to train, so one card. It still trains in its own process, as the
# runs before it did, so that its seeding matches theirs.
if gpu_count >= 1:
    QUEUES = {0: list(TRAINED)}
print(f"{gpu_count} GPU(s):", QUEUES)

print_lock = threading.Lock()
failures = {}


def run_queue(gpu, names):
    for name in names:
        config = configs[name]
        if (config.output_dir / "best.pt").exists():
            with print_lock:
                print(f"[gpu{gpu}] {name}: already trained, skipped")
            continue
        config.output_dir.mkdir(parents=True, exist_ok=True)
        command = [
            sys.executable,
            f"{CHECKOUT}/scripts/train.py",
            "--config",
            f"{CHECKOUT}/configs/phase6/{name}.yaml",
            "--output-dir",
            str(config.output_dir),
            "--num-workers",
            str(config.num_workers),
        ]
        environment = os.environ | {
            "CUDA_VISIBLE_DEVICES": str(gpu),
            "PYTHONPATH": f"{CHECKOUT}/src",
        }
        with print_lock:
            print(f"[gpu{gpu}] {name}: started")
        started = time.perf_counter()
        # The log is kept beside the checkpoint, so it survives in the output
        # even if this cell's printout is truncated.
        with open(config.output_dir / "train.log", "w") as log:
            process = subprocess.Popen(
                command,
                cwd=CHECKOUT,
                env=environment,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
            )
            for line in process.stdout:
                log.write(line)
                log.flush()
                if "Epoch" in line or "Error" in line or "Traceback" in line:
                    with print_lock:
                        print(f"[gpu{gpu}] {name}: {line.rstrip()}")
            exit_code = process.wait()
        minutes = (time.perf_counter() - started) / 60
        with print_lock:
            print(f"[gpu{gpu}] {name}: exit {exit_code} after {minutes:.1f} min")
        if exit_code != 0:
            failures[name] = exit_code


threads = [
    threading.Thread(target=run_queue, args=(gpu, names), daemon=True)
    for gpu, names in QUEUES.items()
]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

for name, exit_code in failures.items():
    log_path = configs[name].output_dir / "train.log"
    print(f"{name} FAILED with exit code {exit_code}; last lines of {log_path}:")
    print("".join(log_path.read_text().splitlines(keepends=True)[-20:]))

In [ ]:
results = {}
for name in RUNS:
    print("=" * 70, name, "=" * 70)
    if name in EARLIER and not (configs[name].output_dir / "best.pt").exists():
        # Earlier runs are candidates only when their output is attached.
        print(f"{name}: not attached, skipped")
        continue
    try:
        results[name] = train_and_score(name)
    except Exception as error:
        # A run that will not fit, or will not build, must not take the others
        # with it. There is no second attempt at this session, so the notebook
        # reaches its end even when one of them does not.
        print(f"{name}: FAILED -- {type(error).__name__}: {error}")
        results[name] = {"failed": f"{type(error).__name__}: {error}"}
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

Path("/kaggle/working/phase6_summary.json").write_text(json.dumps(results, indent=2))

## 4. The comparison

The reference is run *a* of the first round: the same U-Net on ResNet-34, per
annotator targets, 1024, batch 4, twenty epochs, scored through the same
chain. Its figures are quoted from `50_train_kaggle_architectures.ipynb`
(Version 1, commit `2418588`) rather than trained again here.

Adoption needs **+0.01 over run a** and an explanation of why it moved. SQ and
RQ are shown apart because they say different things: SQ is how well a matched
filament is drawn, RQ is how many were matched at all. Run *c* of the first
round scored 0.3309 and 0.3256 on the same seed, so a difference below about
0.005 is within what a rerun alone can produce.

In [ ]:
import pandas as pd

# Run a of the first round, as recorded. Quoted, not recomputed: training it
# again would cost fifty minutes to reproduce four digits already on record.
REFERENCE = {
    "a_unet_resnet34 (round 1)": {
        "architecture": "Unet",
        "encoder": "resnet34",
        "image_size": 1024,
        "batch_size": 4,
        "union_targets": False,
        "pq": 0.3728,
        "sq": 0.6551,
        "rq": 0.5691,
        "tp": 970,
        "fp": 644,
        "fn": 825,
        "fused": 54,
        "split": 77,
        "best_epoch": 15,
        "training_minutes": 47.9,
    }
}
COLUMNS = list(next(iter(REFERENCE.values())))

# Only the runs that produced a score: a failed one carries an explanation
# rather than the columns below.
scored = {name: row for name, row in results.items() if "pq" in row}
for name, row in results.items():
    if "pq" not in row:
        print(f"{name}: {row.get('failed', 'no result')}")

table = pd.DataFrame(REFERENCE | scored).T[COLUMNS]
# A frame built from dictionaries holding both text and numbers comes back as
# text throughout, so the numeric columns are converted before subtracting.
NUMERIC = [
    column for column in COLUMNS if column not in ("architecture", "encoder", "union_targets")
]
table[NUMERIC] = table[NUMERIC].apply(pd.to_numeric, errors="coerce")
table["pq_vs_a"] = (table["pq"] - REFERENCE["a_unet_resnet34 (round 1)"]["pq"]).round(4)
table

## 5. Any maps still missing

Section 3 writes each run's maps as soon as that run is scored, so by now they
normally exist. This fills a gap left by a checkpoint restored from an earlier
version without its maps, and does nothing otherwise. Float16 keeps a run at
1024 to about 300 MB; run *f*'s maps at 2048 are four times that.

In [ ]:
for name in RUNS:
    checkpoint = Path(f"/kaggle/working/{name}/best.pt")
    maps_dir = Path(f"/kaggle/working/prob_fold0_{name}")
    if not checkpoint.exists():
        print(f"{name}: no checkpoint, skipped")
        continue
    if len(list(maps_dir.glob("*.npy"))) == len(val_stems):
        print(f"{name}: maps already written")
        continue

    model, _ = load_checkpoint(checkpoint)
    maps_dir.mkdir(parents=True, exist_ok=True)
    for stem in val_stems:
        probability = predict_probability(
            model,
            load_grayscale(paths.train_images / f"{stem}.jpeg"),
            size=configs[name].image_size,
            device=DEVICE,
        )
        np.save(maps_dir / f"{stem}.npy", probability.astype(np.float16))
    print(f"{name}: {len(list(maps_dir.glob('*.npy')))} maps written")
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 6. Each network at its own post-processing

Section 3 scored every run through one chain, which is what makes them
comparable. **The adoption decision is made on that shared setting.** This
section says whether a run was judged by a gate that was not built for it: the
threshold, the minimum area and the rejoining distance were tuned on the
incumbent, and a network whose probabilities sit elsewhere may lose for that
reason alone. In the first round the retuned figures moved by +0.002 to +0.004
and no ranking changed.

**It runs in the same session as the training** while `SWEEP_WITH_GPU` is
`True`, so one Run All finishes the job. It needs no GPU, though, and a GPU
session charges for its time whether the card is used or not; with
`SWEEP_WITH_GPU = False` a GPU session skips it, and a later run with the
accelerator off does only this.

Run *f* is swept in its own coordinates: the rejoining distances are doubled
and the disk is scaled by 1 rather than 0.5, so each of its settings means the
same thing on the Sun as the matching setting at 1024. Settings are printed in
pixels of each run's map.

As in the first round, the cell prints the sweep and section 3 side by side at
the shared setting. If they disagree, the difference measures the code path,
not the setting, and the retuned figures are not usable until that is settled.

In [ ]:
from filament.data.disk import detect_disk
from filament.postprocess.search import grid, load_maps, sweep

# On: one Run All does everything, at the cost of about an hour and a half of
# GPU quota spent on an idle card. Off leaves the sweep to a CPU-only run.
SWEEP_WITH_GPU = True

retuned = {}
if DEVICE == "cuda" and not SWEEP_WITH_GPU:
    print("GPU pass: sweep skipped. Run again with the accelerator off to sweep.")
else:
    # Found once at full resolution, then scaled per run into its map.
    full_disks = {
        stem: detect_disk(load_grayscale(paths.train_images / f"{stem}.jpeg")) for stem in val_stems
    }

    for name in RUNS:
        config = configs[name]
        maps_dir = Path(f"/kaggle/working/prob_fold0_{name}")
        if len(list(maps_dir.glob("*.npy"))) < len(val_stems):
            print(f"{name}: maps incomplete, skipped")
            continue

        table_path = Path(f"/kaggle/working/sweep_{name}.csv")
        if not table_path.exists():
            found = sorted(Path("/kaggle/input").glob(f"**/sweep_{name}.csv"))
            if found:
                shutil.copy(found[0], table_path)
        if table_path.exists():
            # Swept by an earlier version. The file is written best first.
            done = pd.read_csv(table_path)
            row = done.iloc[0]
            retuned[name] = {
                "setting": {
                    key: row[key]
                    for key in ("threshold", "min_area", "join_gap", "join_offset")
                    if key in done
                },
                "pq": round(float(row["pq"]), 4),
                "sq": round(float(row["sq"]), 4),
                "rq": round(float(row["rq"]), 4),
                "tp": int(row["tp"]),
                "fp": int(row["fp"]),
                "fn": int(row["fn"]),
            }
            print(f"{name}: already swept, best {row['pq']:.4f}")
            continue

        scale = config.image_size / SCORING_SIZE
        settings = grid(
            threshold=[0.3, 0.4, 0.5, 0.6, 0.7],
            min_area=[200, 400, 600],
            join_gap=[0.0, SCORING["join_gap"] * scale],
            join_offset=[DEFAULT_MAX_OFFSET * scale],
        )
        disks = {stem: disk.scaled(config.image_size / 2048) for stem, disk in full_disks.items()}

        # The whole body, not just the sweep: one run that cannot be read or
        # written must not cost the others.
        try:
            maps = load_maps(maps_dir, val_stems)
            outcome = sweep(maps, gt_df, settings, disks=disks)
            outcome.table.to_csv(table_path, index=False)

            shared = [
                point for point in outcome.points if point.setting.values == scoring_for(config)
            ]
            if shared and "pq" in results.get(name, {}):
                print(
                    f"{name}: same setting -- sweep {shared[0].pq.pq:.4f}, "
                    f"section 3 {results[name]['pq']:.4f}"
                )

            best = outcome.best
            retuned[name] = {
                "setting": dict(best.setting.values),
                "pq": round(best.pq.pq, 4),
                "sq": round(best.pq.sq, 4),
                "rq": round(best.pq.rq, 4),
                "tp": best.pq.tp,
                "fp": best.pq.fp,
                "fn": best.pq.fn,
            }
            print(f"{name}: best {best.pq.pq:.4f} at {best.setting}")
            del maps
        except Exception as error:
            print(f"{name}: FAILED -- {type(error).__name__}: {error}")

    Path("/kaggle/working/phase6_retuned.json").write_text(json.dumps(retuned, indent=2))

## 7. The submission

Written from the candidate with the best PQ at the shared setting of section 3,
through that same setting. Not from the sweep's best: the adoption rule is read
at the shared setting, and in every sweep so far the best point beat it by
0.005 or less, which is within what a rerun alone moves.

The overlap check is not optional. Kaggle rejects a submission whose masks
share a pixel, and the rejected attempt still counts against the five allowed
per day.

Do not submit to explore. The local PQ decides; the leaderboard is where a
decision already made is checked.

In [ ]:
from filament.evaluation import predict_frame
from filament.metrics.overlap import check_no_overlap
from filament.postprocess.instances import instances_to_rows
from filament.submit.rle import write_submission

candidates = {name: row for name, row in results.items() if "pq" in row}
if not candidates:
    raise SystemExit("No run was scored, so there is nothing to submit.")
chosen = max(candidates, key=lambda name: candidates[name]["pq"])
chosen_config = configs[chosen]
chosen_scoring = scoring_for(chosen_config)
print(f"submitting {chosen}: fold 0 PQ {candidates[chosen]['pq']} at {chosen_scoring}")
for name, row in sorted(candidates.items(), key=lambda item: -item[1]["pq"]):
    print(f"  {name:<24} {row['pq']}")

model, _ = load_checkpoint(chosen_config.output_dir / "best.pt")
test_stems = sorted(path.stem for path in paths.test_images.glob("*.jpeg"))
rows = []
empty = []
for position, stem in enumerate(test_stems, start=1):
    instances = predict_frame(
        model,
        paths.test_images / f"{stem}.jpeg",
        size=chosen_config.image_size,
        device=DEVICE,
        **chosen_scoring,
    )
    if not instances:
        empty.append(stem)
    rows.extend(instances_to_rows(stem, instances))
    if position % 30 == 0:
        print(f"{position}/{len(test_stems)}")

submission_path = Path("/kaggle/working/submission.csv")
write_submission(rows, submission_path)
check_no_overlap(submission_path)
print(f"{len(rows)} masks over {len(test_stems)} frames; overlap check passed")
if empty:
    # Every training frame holds at least one filament, so an empty frame is a
    # miss rather than an answer.
    print(f"{len(empty)} frame(s) with no prediction: {', '.join(empty[:5])}")

Path("/kaggle/working/submission_info.json").write_text(
    json.dumps(
        {
            "run": chosen,
            "commit": REF,
            "fold0": candidates[chosen],
            "scoring": chosen_scoring,
            "test_frames": len(test_stems),
            "masks": len(rows),
            "empty_frames": empty,
        },
        indent=2,
    )
)
del model
if DEVICE == "cuda":
    torch.cuda.empty_cache()
pd.read_csv(submission_path).head()

## 8. What to record

Into the lab notebook, for every run and not only the winner — the ones that
did not work are what the ablation table in the final report is made of:

- PQ, SQ, RQ, TP, FP, FN, and the fused and split counts
- best epoch and training time, noting that runs shared the session's two cards
  and so the minutes are not comparable with the first round's
- that runs *e* and *h* trained at batch 2 and 1, against 4 for the rest
- the commit hash this notebook cloned

- which run `submission.csv` came from (`submission_info.json`), and its
  leaderboard score next to the fold 0 PQ once it is submitted

Folds 1 to 4 stay untouched until a configuration is frozen.